# 10 - Generate prediction - gemma2

### Setup

In [1]:
# ==== SETUP ====
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton unsloth_zoo
!pip install --no-deps cut_cross_entropy unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 MB 39.1 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Konfigurasi

In [3]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except ImportError:
        token = os.environ.get("HF_Faiss")
        if token is None:
            raise RuntimeError("HF_TOKEN tidak ditemukan.")
        return token

HF_TOKEN = get_hf_token()
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "/workspace")

MODEL_KEY = "gemma2"
BASE_MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"
ADAPTER_REPO = f"Makaareeem/{MODEL_KEY}-qlora-adapter"
SYSTEM_ROLE = False
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS "
    "berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum menjawab."
)

## 2. Load test set

In [4]:
test_ds = load_dataset("csv", data_files=f"hf://datasets/{HF_REPO}/test.csv", split="train", token=HF_TOKEN)
print(f"test set: {len(test_ds)} baris")

test.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

test set: 1791 baris


In [5]:
test_ds.to_pandas()

,record_id,task_type,subtask_type,split,instruction,cot,response,context_joined,n_context_spans,document_title,document_year,section_id,section_title,context_chunk_ids,source_pages,bps_url,track_a_score,numeric_score
0,b05d48cbd7388514,qa,factual_retrieval,test,Apa rekomendasi pertama dari KPK yang disusun ...,"Teks menyatakan ""Pertama, diperlukan pengemban...",Rekomendasi pertama KPK adalah pengembangan da...,"Pertama, diperlukan pengembangan dan evaluasi ...",1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_073,3.3 Menciptakan Ekosistem Pendidikan tanpa Kor...,analisis-petty-corruption-hasil-spak-2020---20...,102 ||| 103 ||| 104,https://www.bps.go.id/id/publication/2025/12/3...,1.000,1.0
1,6fd9fbd0051e170c,qa,factual_retrieval,test,Siapa yang memprakarsai GONE theory dan apa sa...,"Teks menyatakan ""GONE theory yang diprakarsai ...",GONE theory diprakarsai oleh Jack Bologne. Emp...,GONE theory yang diprakarsai Jack Bologne menj...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_073,3.3 Menciptakan Ekosistem Pendidikan tanpa Kor...,analisis-petty-corruption-hasil-spak-2020---20...,102 ||| 103 ||| 104,https://www.bps.go.id/id/publication/2025/12/3...,1.000,1.0
2,9b46bc8bdc31522e,qa,factual_retrieval,test,Menurut pendekatan lingkungan maupun kultural ...,"Teks menyatakan ""Dalam pendekatan lingkungan m...",Solusi yang dapat diimplementasikan dalam pend...,Dalam pendekatan lingkungan maupun kultural ma...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_073,3.3 Menciptakan Ekosistem Pendidikan tanpa Kor...,analisis-petty-corruption-hasil-spak-2020---20...,102 ||| 103 ||| 104,https://www.bps.go.id/id/publication/2025/12/3...,1.000,1.0
3,7e0baf29d0fa19e6,qa,factual_retrieval,test,Sebutkan empat langkah pencegahan khusus korup...,"Teks menyatakan ""Heyneman (2004) menyatakan pa...",Empat langkah pencegahan khusus korupsi pada s...,Pertama adalah diperlukan reformasi struktural...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_073,3.3 Menciptakan Ekosistem Pendidikan tanpa Kor...,analisis-petty-corruption-hasil-spak-2020---20...,102 ||| 103 ||| 104,https://www.bps.go.id/id/publication/2025/12/3...,0.535,0.0
4,8eaf6490b590c374,qa,definition_clarification,test,Apa yang dimaksud dengan GONE theory dalam kon...,"Teks menyatakan ""GONE theory yang diprakarsai ...",GONE theory adalah teori yang diprakarsai Jack...,GONE theory yang diprakarsai Jack Bologne menj...,1,Analisis Petty Corruption Hasil SPAK 2020–2024,2025,sec_073,3.3 Menciptakan Ekosistem Pendidikan tanpa Kor...,analisis-petty-corruption-hasil-spak-2020---20...,102 ||| 103 ||| 104,https://www.bps.go.id/id/publication/2025/12/3...,1.000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1786,03460d992d3fd3a0,qa,methodology_explanation,test,Apa sumber data yang digunakan untuk menghasil...,"Teks menyatakan ""Badan Pusat Statistik, Survei...",Data bersumber dari Badan Pusat Statistik mela...,"Sumber/ Source : Badan Pusat Statistik, Survei...",1,Statistik Kesejahteraan Rakyat 2024,2024,sec_169,Sampling Error Persentase Penduduk yang Mempun...,statistik-kesejahteraan-rakyat-2024__sec_169__...,334 ||| 335 ||| 336,https://www.bps.go.id/id/publication/2024/10/2...,1.000,1.0
1787,3f25cc3863855ead,qa,definition_clarification,test,Apa yang diukur oleh indikator 'Persentase Pen...,"Teks menyatakan indikator tersebut adalah ""Per...",Indikator ini mengukur proporsi penduduk yang ...,Persentase Penduduk yang Pernah Rawat Inap dal...,1,Statistik Kesejahteraan Rakyat 2024,2024,sec_169,Sampling Error Persentase Penduduk yang Mempun...,statistik-kesejahteraan-rakyat-2024__sec_169__...,334 ||| 335 ||| 336,https://www.bps.go.id/id/publication/2024/10/2...,1.000,1.0
1788,94755286a0fca43d,qa,source_navigation,test,Di bagian mana data mengenai akses rumah tangg...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",Data akses rumah tangga terhadap fasilitas san...,Lanjutan Tabel / Continued Table 9.6.3,1,Statistik Kesejahteraan Rak

## 3. Format chat

In [6]:
# Susun user content (instruction + context_joined), sama seperti saat training
def build_user_content(example):
    return f"Konteks:\n{example['context_joined']}\n\nPertanyaan: {example['instruction']}"


# Susun ground truth (cot + response), PERSIS sama seperti target training
def build_ground_truth(example):
    return f"{example['cot']}\n\n{example['response']}"


# Susun chat messages untuk prompt generate, tanpa assistant turn (biar model yang isi)
def build_prompt_messages(example, system_role):
    user_content = build_user_content(example)
    if system_role:
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
    return [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_content}"}]

## 4. Fungsi Generate

In [7]:
def generate_batch(model, tokenizer, examples, system_role, batch_size=8):
    FastLanguageModel.for_inference(model)
    predictions = []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                build_prompt_messages(ex, system_role),
                tokenize=False,
                add_generation_prompt=True,
            )
            for ex in batch
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
        decoded = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        predictions.extend(decoded)
        print(f"generate {i + len(batch)}/{len(examples)}")
    return predictions

### Base model

In [8]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

examples = [test_ds[i] for i in range(len(test_ds))]
base_predictions = generate_batch(base_model, base_tokenizer, examples, SYSTEM_ROLE)

del base_model, base_tokenizer
import torch
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.9.3: Fast Gemma2 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.042 GiB
no_split classes   : ['Gemma2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.494 GiB
activation reserve : 11.706 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  0.943 GiB  free 12.009 GiB  reserve 11.706 GiB
  cuda:1  budget  13.00 GiB  weights  1.099 GiB  free 11.898 GiB  reserve 11.403 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.440 -> 12.996 GiB (memory t

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


generate 8/1791
generate 16/1791
generate 24/1791
generate 32/1791
generate 40/1791
generate 48/1791
generate 56/1791
generate 64/1791
generate 72/1791
generate 80/1791
generate 88/1791
generate 96/1791
generate 104/1791
generate 112/1791
generate 120/1791
generate 128/1791
generate 136/1791
generate 144/1791
generate 152/1791
generate 160/1791
generate 168/1791
generate 176/1791
generate 184/1791
generate 192/1791
generate 200/1791
generate 208/1791
generate 216/1791
generate 224/1791
generate 232/1791
generate 240/1791
generate 248/1791
generate 256/1791
generate 264/1791
generate 272/1791
generate 280/1791
generate 288/1791
generate 296/1791
generate 304/1791
generate 312/1791
generate 320/1791
generate 328/1791
generate 336/1791
generate 344/1791
generate 352/1791
generate 360/1791
generate 368/1791
generate 376/1791
generate 384/1791
generate 392/1791
generate 400/1791
generate 408/1791
generate 416/1791
generate 424/1791
generate 432/1791
generate 440/1791
generate 448/1791
gener

### Finetuned model

In [9]:
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_REPO,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

ft_predictions = generate_batch(ft_model, ft_tokenizer, examples, SYSTEM_ROLE)

del ft_model, ft_tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.9.3: Fast Gemma2 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.042 GiB
no_split classes   : ['Gemma2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.494 GiB
activation reserve : 10.570 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  11.72 GiB  weights  0.943 GiB  free 10.780 GiB  reserve 10.570 GiB
  cuda:1  budget  11.95 GiB  weights  1.099 GiB  free 10.853 GiB  reserve 10.359 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 13.026 -> 11.724 GiB, cuda:1 13.280 -> 11.952 GiB (memory t

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.3 patched 26 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


generate 8/1791
generate 16/1791
generate 24/1791
generate 32/1791
generate 40/1791
generate 48/1791
generate 56/1791
generate 64/1791
generate 72/1791
generate 80/1791
generate 88/1791
generate 96/1791
generate 104/1791
generate 112/1791
generate 120/1791
generate 128/1791
generate 136/1791
generate 144/1791
generate 152/1791
generate 160/1791
generate 168/1791
generate 176/1791
generate 184/1791
generate 192/1791
generate 200/1791
generate 208/1791
generate 216/1791
generate 224/1791
generate 232/1791
generate 240/1791
generate 248/1791
generate 256/1791
generate 264/1791
generate 272/1791
generate 280/1791
generate 288/1791
generate 296/1791
generate 304/1791
generate 312/1791
generate 320/1791
generate 328/1791
generate 336/1791
generate 344/1791
generate 352/1791
generate 360/1791
generate 368/1791
generate 376/1791
generate 384/1791
generate 392/1791
generate 400/1791
generate 408/1791
generate 416/1791
generate 424/1791
generate 432/1791
generate 440/1791
generate 448/1791
gener

## 5. Simpan Hasil

In [10]:
# ==== SIMPAN HASIL SEBAGAI CSV (ans_ref, ans_pred_base, ans_pred_finetuned) ====
results_df = pd.DataFrame({
    "record_id": [ex.get("record_id", i) for i, ex in enumerate(examples)],
    "instruction": [ex["instruction"] for ex in examples],
    "ans_ref": [build_ground_truth(ex) for ex in examples],
    "ans_pred_base": base_predictions,
    "ans_pred_finetuned": ft_predictions,
})

output_path = f"{OUTPUT_DIR}/eval_{MODEL_KEY}_predictions.csv"
results_df.to_csv(output_path, index=False)
print(f"Tersimpan: {output_path}")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=output_path,
    path_in_repo=f"eval_{MODEL_KEY}_predictions.csv",
    repo_id=HF_REPO,
    repo_type="dataset",
)

Tersimpan: /kaggle/working/eval_gemma2_predictions.csv


CommitInfo(commit_url='https://huggingface.co/datasets/Makaareeem/publikasi-rag-finetuning-dataset/commit/f5b1720e21ffa2ba554135f0a616337031eb8bcc', commit_message='Upload eval_gemma2_predictions.csv with huggingface_hub', commit_description='', oid='f5b1720e21ffa2ba554135f0a616337031eb8bcc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Makaareeem/publikasi-rag-finetuning-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Makaareeem/publikasi-rag-finetuning-dataset'), pr_revision=None, pr_num=None)